# Dunnhumby: RBF 이후 지도도 게이트 M2 (개발 seed 43)

사용자 N/V를 모두 유지하고, 상품 가격은 세부유형→대분류→전체가격으로 backoff한 하나의 백분위를 사용합니다. 이번 수정은 가격 백분위를 RBF 표현으로 변환하고 중심화한 뒤 상품별 학습 구매고객 지지도 `r=n/(n+10)`를 곱합니다: `x_i^P=r_i[φ(p_i)−μ_P]`. 따라서 `r=0`인 상품의 가격 개입은 정확히 0이 됩니다. 기존 M1은 검증해 재사용하고 새 M2 한 arm만 학습합니다. binary graph·uniform K=1·plain BPR·한 optimizer, 최대 300 epoch/25 epoch 간격 평가이며 final test/holdout이 아닙니다.


In [ ]:
from pathlib import Path
import sys, subprocess, json, shutil
from google.colab import drive, files
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_nv_modulated_id_seed43_v3/reports/result.json'
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not ROOT.is_dir() or not REPORT.is_file():
        raise RuntimeError(f'기준 M1 결과를 확인하세요: {REPORT}')
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결 실패. 이 셀부터 다시 실행하세요. 학습은 시작하지 않았습니다.') from exc
SOURCE_COMMIT = '81ea0df517210cb328d315808cb5c4de5b216f6a'
REPO = Path('/content/clv-m2-reliability-gated-price-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
import clv_m2_reliability_gated_price_screen as screen
OUT = ROOT/'results_v3_dunnhumby_m2_reliability_gated_price_seed43_v1'
print('source:', SOURCE_COMMIT, '| report:', REPORT, '| out:', OUT)


In [ ]:
cfg, prepared = screen.prepare(REPORT, OUT)
print(json.dumps({
    'code_version': screen.VERSION, 'seed': 43, 'epochs': cfg.epochs,
    'new_arms': [screen.spec()], 'feature_settings': screen.SETTINGS,
    'final_test': False, 'holdout': False
}, ensure_ascii=False, indent=2))


In [ ]:
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd
absolute = pd.read_csv(paths['absolute'])
comparison = pd.read_csv(paths['comparison'])
diagnostics = pd.read_csv(paths['diagnostics'])
report = json.loads(Path(paths['json']).read_text())
print('1) 전체 절대지표')
display(absolute.T)
print('2) M2−M1 전체 비교')
display(comparison)
print('3) 작동 진단')
display(diagnostics)
print('4) 사전 판정')
print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
archive = shutil.make_archive('/content/m2_reliability_gated_price_seed43_results', 'zip', OUT)
files.download(archive)
